# 🍽️ Weekly Themed Meal Planner MVP

A meal planning assistant that generates 7 themed dinners with consolidated shopping lists.

## Features
- 7 unique themed dinners (Monday-Sunday)
- 5 ingredients per recipe (excluding basics)
- Dietary preference support (vegan, vegetarian, keto, omnivore)
- Consolidated shopping list
- Email delivery

## 1. Installation & Setup

In [1]:
%pip install -q langchain langchain-openai langchain-anthropic pydantic gradio python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

Create a file called `.env` in the same folder as this notebook with this line:

```
OPENAI_API_KEY=sk-...
```

**Edit the values below before running the notebook.**

In [1]:
# ============================================================
# CONFIGURATION - Edit these values
# ============================================================
import os
from dotenv import load_dotenv

load_dotenv(".env")

# API Keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Model settings
OPENAI_MODEL = "gpt-4o-mini"
TEMPERATURE = 0.7

# Email settings (Gmail with App Password)
EMAIL_SENDER = "provide your email"
EMAIL_PASSWORD = "password"

# Meal generation settings
INGREDIENTS_PER_RECIPE = 5
EXCLUDED_BASICS = ["salt", "pepper", "oil", "garlic", "water"]
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

In [2]:
# Set environment variables
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

**Baseline Test – Missing Environment Configuration**

I temporarily renamed the `.env` file and restarted the Jupyter kernel to test how the original configuration handles a missing API key.

The configuration failed at `os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY` with the following error:

`TypeError: str expected, not NoneType`

The issue occurs because `os.getenv("OPENAI_API_KEY")` returns `None` when the API key is unavailable. The original code does not validate this value before assigning it to an environment variable.

The error message indicates a type mismatch but does not clearly explain that the API key is missing.

**Baseline Review – Hardcoded Configuration Values**

The configuration cell contains hardcoded email settings. While the current values are placeholders, real credentials such as `EMAIL_PASSWORD` should never be stored directly in a notebook or pushed to GitHub. The OpenAI API key is already loaded from a `.env` file, which is a safer approach as long as the file is excluded from version control. Other settings, such as the model name, temperature, and meal generation parameters, are not sensitive and can remain in the notebook.

**Baseline Test – Successful Meal Plan Generation**

I successfully launched the original Gradio application and generated a weekly meal plan using the default omnivore diet without additional preferences.

The application generated seven unique themed dinners from Monday to Sunday, each listing five ingredients. It also created a consolidated shopping list, correctly combining repeated ingredients such as chicken breast and eggs.

The main workflow from user input to generated output works as expected. However, some basic ingredients, such as black pepper and olive oil, still appear in the shopping list despite the exclusion settings. This is worth reviewing during the refactoring process.

**Baseline Test – Email Functionality**

I tested the Email Plan feature in the Gradio interface using the original placeholder email settings. The email could not be sent, and the application displayed the following message:

`Failed to send email: Connection unexpectedly closed`

The existing exception handling prevented the application from crashing and displayed the error in the status field. However, the message does not clearly identify the underlying issue or explain how to resolve it.

This shows that the email functionality requires further review during refactoring, particularly regarding configuration validation and error handling.

**Baseline Review – Dependencies**

I reviewed the installation requirements and searched the notebook for package imports. The application uses `langchain-openai` and components from `langchain-core`, but I found no direct usage of `langchain` or `langchain-anthropic`.

Since `langchain-core` is already installed as a dependency of `langchain-openai`, the separate installation of `langchain` appears unnecessary. The `langchain-anthropic` package is also not needed because the application uses OpenAI rather than Anthropic models.

These unnecessary dependencies can be removed during refactoring to simplify the setup.

### Problem #1: Hardcoded Email Credentials

**Category:** Hardcoded Values / Security  
**Priority:** High

**Problem:**  
The email sender and password are hardcoded in the configuration cell. While the current values are placeholders, replacing them with real credentials could expose sensitive information if the notebook is shared or pushed to GitHub.

**Potential Improvement:**  
Store the email credentials in the existing `.env` file, similar to the OpenAI API key. Make sure the `.env` file is included in `.gitignore` to prevent sensitive information from being committed to GitHub.

## 3. Pydantic Models for Validation

In [3]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
from enum import Enum


class DietaryPreference(str, Enum):
    OMNIVORE = "omnivore"
    VEGETARIAN = "vegetarian"
    VEGAN = "vegan"
    KETO = "keto"


class Ingredient(BaseModel):
    """Single ingredient with quantity."""
    name: str = Field(..., description="Name of the ingredient")
    quantity: str = Field(..., description="Quantity with unit (e.g., '2 cups', '500g')")


class Recipe(BaseModel):
    """A single themed dinner recipe."""
    day: str = Field(..., description="Day of the week")
    theme: str = Field(..., description="Cuisine theme (e.g., 'Italian', 'Mexican')")
    dish_name: str = Field(..., description="Name of the dish")
    ingredients: List[Ingredient] = Field(..., description="List of 5 ingredients")
    instructions: str = Field(..., description="Brief cooking instructions")

    @field_validator('ingredients')
    @classmethod
    def validate_ingredient_count(cls, v):
        if len(v) != INGREDIENTS_PER_RECIPE:
            raise ValueError(f"Recipe must have exactly {INGREDIENTS_PER_RECIPE} ingredients, got {len(v)}")
        return v


class ShoppingItem(BaseModel):
    """Consolidated shopping list item."""
    name: str = Field(..., description="Ingredient name")
    total_quantity: str = Field(..., description="Total quantity needed")
    used_in: List[str] = Field(..., description="List of dishes using this ingredient")


class MealPlan(BaseModel):
    """Complete weekly meal plan."""
    dietary_preference: str = Field(..., description="Selected dietary preference")
    recipes: List[Recipe] = Field(..., description="List of 7 recipes for the week")
    shopping_list: List[ShoppingItem] = Field(..., description="Consolidated shopping list")

    @field_validator('recipes')
    @classmethod
    def validate_recipe_count(cls, v):
        if len(v) != 7:
            raise ValueError(f"Meal plan must have exactly 7 recipes, got {len(v)}")
        return v

    @field_validator('recipes')
    @classmethod
    def validate_unique_themes(cls, v):
        themes = [r.theme.lower() for r in v]
        if len(themes) != len(set(themes)):
            raise ValueError("All 7 themes must be unique")
        return v

## 4. LangChain Setup & Prompts

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser


def get_llm():
    """Get the configured LLM."""
    return ChatOpenAI(model=OPENAI_MODEL, temperature=TEMPERATURE)


# Main meal plan generation prompt
MEAL_PLAN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a professional meal planner. Generate a weekly meal plan with EXACTLY 7 themed dinners.

STRICT RULES - FOLLOW EXACTLY:
1. Each recipe MUST have EXACTLY 5 main ingredients (do NOT count salt, pepper, oil, garlic, or water)
2. ALL 7 THEMES MUST BE COMPLETELY DIFFERENT - no duplicates allowed! Use: Italian, Mexican, Japanese, Indian, Thai, Greek, American, Chinese, French, Korean, Mediterranean, Middle Eastern, etc.
3. All ingredients MUST be real, commonly available items
4. Follow the dietary preference strictly
5. Provide realistic quantities for 2 servings

Dietary Guidelines:
- Omnivore: Any ingredients allowed
- Vegetarian: No meat or fish, eggs and dairy OK
- Vegan: No animal products at all
- Keto: Low carb, high fat, no grains/sugar/starchy vegetables

Output valid JSON matching this exact structure:
{format_instructions}"""),
    ("human", """Create a weekly meal plan:
- Diet: {dietary_preference}
- Theme requests: {theme_requests}
- Preferences: {additional_preferences}

IMPORTANT: Each day MUST have a DIFFERENT cuisine theme. Generate 7 unique themed dinners for Monday-Sunday.""")
])


# Shopping list consolidation prompt
SHOPPING_LIST_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a shopping list organizer. Consolidate ingredients from multiple recipes into a single shopping list.

RULES:
1. Combine same ingredients and sum their quantities
2. List which dishes use each ingredient
3. Use standard units (cups, tablespoons, grams, pieces)
4. Round up quantities for convenience

Output valid JSON matching this structure:
{format_instructions}"""),
    ("human", """Consolidate the shopping list from these recipes:
{recipes_json}""")
])

### Problem #9: Unused Code

**Category:** Dead Code / Maintainability  
**Priority:** Low

**Problem:**  
The `SHOPPING_LIST_PROMPT` and `self.shopping_parser` are defined but never used in the application.

The shopping list is already generated together with the recipes through the `MEAL_PLAN_PROMPT`, which uses the JSON format instructions from the `MealPlan` Pydantic model.

These unused components add unnecessary complexity and can make the code harder to understand and maintain.

**Potential Improvement:**  
Remove the unused shopping list prompt and parser if they are not needed for the intended workflow. This would simplify the code without changing the existing functionality.

## 5. Meal Generation Logic

In [5]:
import json
import time


class MealPlanGenerator:
    """Generates weekly themed meal plans."""

    def __init__(self):
        self.llm = get_llm()
        self.recipe_parser = JsonOutputParser(pydantic_object=MealPlan)
        self.shopping_parser = JsonOutputParser(pydantic_object=ShoppingItem)

    def generate(
        self,
        dietary_preference: str,
        theme_requests: str = "random",
        additional_preferences: str = "none"
    ) -> dict:
        """Generate a complete meal plan with shopping list."""
        start_time = time.time()

        recipe_chain = MEAL_PLAN_PROMPT | self.llm

        result = recipe_chain.invoke({
            "dietary_preference": dietary_preference,
            "theme_requests": theme_requests if theme_requests else "random diverse cuisines",
            "additional_preferences": additional_preferences if additional_preferences else "none",
            "format_instructions": self.recipe_parser.get_format_instructions()
        })

        try:
            content = result.content
            if "```json" in content:
                content = content.split("```json")[1].split("```")[0]
            elif "```" in content:
                content = content.split("```")[1].split("```")[0]

            meal_plan_data = json.loads(content)
            meal_plan = MealPlan(**meal_plan_data)
            generation_time = time.time() - start_time

            return {
                "success": True,
                "meal_plan": meal_plan.model_dump(),
                "generation_time": round(generation_time, 2),
                "error": None
            }

        except json.JSONDecodeError as e:
            return {"success": False, "meal_plan": None, "error": f"JSON parsing error: {str(e)}"}
        except Exception as e:
            return {"success": False, "meal_plan": None, "error": str(e)}


# Initialize generator
generator = MealPlanGenerator()

### Problem #2: Monolithic Function

**Category:** Monolithic Function / Code Structure  
**Priority:** Medium

**Problem:**  
The `MealPlanGenerator.generate()` method handles several different responsibilities, including input preparation, LLM communication, response cleaning, JSON parsing, Pydantic validation, execution time measurement, error handling, and returning the results.

Combining these tasks in a single method makes the code harder to understand, maintain, and test independently.

**Potential Improvement:**  
Separate the main responsibilities into smaller, reusable functions. For example, the LLM interaction could be separated from response processing and validation. This would make it easier to test individual components without calling the OpenAI API and help identify errors more precisely.

### Problem #3: Missing Error Handling for API Calls

**Category:** Missing Error Handling  
**Priority:** High

**Problem:**  
The `recipe_chain.invoke()` API call is located outside the `try` block in `MealPlanGenerator.generate()`. If the API request fails, for example due to an invalid API key or a connection issue, the existing exception handlers cannot catch the error.

As a result, the method may terminate unexpectedly instead of returning a structured error response.

**Potential Improvement:**  
Include the API call within the error-handling logic to catch authentication errors, connection issues, and other API-related exceptions. This would allow the application to return a controlled error response instead of propagating an unhandled exception.

### Problem #4: Insufficient Error Information

**Category:** Silent Failure / Error Handling  
**Priority:** Medium

**Problem:**  
The `generate()` method catches JSON parsing errors and other exceptions, but the error responses do not consistently identify where the failure occurred. The general exception handler only returns `str(e)` without recording the processing step or a traceback.

This can make debugging more difficult because the error message may describe what went wrong without clearly identifying where it happened.

**Potential Improvement:**  
Improve error reporting by adding context about the processing step and using logging to record technical error details. This would make it easier to identify the source of a failure while keeping error messages understandable.

### Problem #5: Fragile JSON Processing

**Category:** Fragile Logic / JSON Parsing  
**Priority:** Medium

**Problem:**  
The `generate()` method uses string splitting to remove Markdown code fences from the LLM response. While this works for some common formats, it is not reliable for all possible outputs.

For example, if the model includes an introductory sentence before the JSON without using code fences, `json.loads()` will fail even if the response contains valid JSON data.

**Potential Improvement:**  
Use a more robust approach to extract and parse structured JSON responses. This would reduce parsing errors caused by unexpected formatting in the LLM output.

### Problem #6: Inconsistent Output Structure

**Category:** Inconsistent Output / Code Structure  
**Priority:** Medium

**Problem:**  
The `generate()` method returns dictionaries with different keys depending on whether the generation succeeds or fails. The `generation_time` key is included in successful responses but missing from error responses.

The `generate_plan_streaming()` function accesses this key to display the generation time. However, it checks the `success` value first and stops processing if generation fails, so the missing key does not currently cause an error in this workflow.

Nevertheless, the inconsistent structure could cause problems if other functions access the missing key without checking the result first.

**Potential Improvement:**  
Use a consistent dictionary structure for both successful and failed generations. For example, include `generation_time` with a value of `None` when generation fails.

## 6. Email Functionality

In [6]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart


def format_meal_plan_email(meal_plan: dict) -> str:
    """Format meal plan as readable email content (shopping list first)."""
    lines = []
    lines.append("=" * 40)
    lines.append(f"WEEKLY MEAL PLAN ({meal_plan['dietary_preference'].upper()})")
    lines.append("=" * 40)
    lines.append("")

    # Shopping list FIRST
    lines.append("SHOPPING LIST")
    lines.append("-" * 20)
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} {item['name']}")
    lines.append("")

    # Recipes
    lines.append("=" * 40)
    lines.append("RECIPES")
    lines.append("=" * 40)
    for recipe in meal_plan['recipes']:
        lines.append(f"\n{recipe['day'].upper()} - {recipe['theme']}")
        lines.append(f"{recipe['dish_name']}")
        ingredients = ", ".join([f"{ing['quantity']} {ing['name']}" for ing in recipe['ingredients']])
        lines.append(f"Ingredients: {ingredients}")
        lines.append(f"Instructions: {recipe['instructions']}")

    return "\n".join(lines)


def send_meal_plan_email(recipient_email: str, meal_plan: dict) -> dict:
    """Send meal plan via email."""
    try:
        msg = MIMEMultipart()
        msg['From'] = EMAIL_SENDER
        msg['To'] = recipient_email
        msg['Subject'] = "Your Weekly Meal Plan"

        body = format_meal_plan_email(meal_plan)
        msg.attach(MIMEText(body, 'plain'))

        with smtplib.SMTP('smtp.gmail.com', 587) as server:
            server.starttls()
            server.login(EMAIL_SENDER, EMAIL_PASSWORD)
            server.send_message(msg)

        return {"success": True, "message": f"Email sent to {recipient_email}"}
    except Exception as e:
        return {"success": False, "message": f"Failed to send email: {str(e)}"}

### Problem #7: Duplicated Ingredient Formatting

**Category:** Repeated Code / Maintainability  
**Priority:** Low

**Problem:**  
The `format_meal_plan_email()` and `format_single_recipe()` functions contain the same logic for converting recipe ingredients into a formatted string.

This duplication makes maintenance more difficult because changes to the ingredient formatting have to be applied in both places. If one function is updated and the other is not, the formatting could become inconsistent.

**Potential Improvement:**  
Extract the shared ingredient formatting logic into a reusable helper function. Both functions could use this helper while keeping their individual output formats.

### Problem #12: Mixed Responsibilities in Email Functionality

**Category:** Mixed Concerns / Code Structure  
**Priority:** Medium

**Problem:**  
The `send_meal_plan_email()` function handles both email message construction and SMTP communication. It prepares the message, including sender, recipient, subject, and content, before connecting to the Gmail server and attempting to send it.

Combining these responsibilities makes it harder to test message creation independently from the actual email delivery process.

**Potential Improvement:**  
Separate email message construction from SMTP communication. This would allow the application to test the email content without requiring valid Gmail credentials or an active SMTP connection. It would also make debugging easier by separating formatting issues from delivery errors.

## 7. Validation

In [7]:
def run_basic_validation(meal_plan_result: dict) -> dict:
    """Run quick programmatic validation checks (not LLM-based)."""
    if not meal_plan_result.get("success"):
        return {"passed": False, "errors": [meal_plan_result.get("error")]}

    meal_plan = meal_plan_result["meal_plan"]
    errors = []

    # Check 1: 7 recipes
    if len(meal_plan["recipes"]) != 7:
        errors.append(f"Expected 7 recipes, got {len(meal_plan['recipes'])}")

    # Check 2: Unique themes
    themes = [r["theme"].lower() for r in meal_plan["recipes"]]
    if len(themes) != len(set(themes)):
        duplicates = [t for t in themes if themes.count(t) > 1]
        errors.append(f"Duplicate themes: {set(duplicates)}")

    # Check 3: 5 ingredients per recipe
    for recipe in meal_plan["recipes"]:
        if len(recipe["ingredients"]) != 5:
            errors.append(f"{recipe['day']}: {len(recipe['ingredients'])} ingredients (expected 5)")

    return {"passed": len(errors) == 0, "errors": errors}

### Problem #10: Hardcoded Validation Values

**Category:** Hardcoded Values / Maintainability  
**Priority:** Medium

**Problem:**  
The `run_basic_validation()` function uses hardcoded values of `5` for ingredients per recipe and `7` for the number of recipes, even though these values are already defined in the configuration.

This can lead to inconsistent validation results if the configuration is changed without updating the validation logic. For example, changing the required ingredient count to six would still cause the function to expect five ingredients.

**Potential Improvement:**  
Replace the hardcoded values with the existing configuration variables `INGREDIENTS_PER_RECIPE` and `len(DAYS)`. This would make the validation logic more consistent and easier to maintain.

### Problem #11: Duplicated Validation Logic

**Category:** Duplicated Logic / Maintainability  
**Priority:** Medium

**Problem:**  
The application validates several rules in two different places: the Pydantic models and the `run_basic_validation()` function. Both check the number of recipes, unique themes, and the number of ingredients per recipe.

Although multiple validation layers can provide additional protection, duplicating the same rules creates unnecessary maintenance effort and may lead to inconsistent results.

For example, if `INGREDIENTS_PER_RECIPE` is changed from 5 to 6, Pydantic would accept six ingredients while `run_basic_validation()` would still expect five.

**Potential Improvement:**  
Avoid duplicating identical validation rules or ensure that both validation layers use the same configuration values. Pydantic could handle structural validation, while additional checks could focus on rules not already covered.

## 8. Gradio Interface

In [8]:
import gradio as gr
import time as pytime


def format_shopping_list(meal_plan: dict) -> str:
    """Format shopping list section."""
    lines = ["## Shopping List\n"]
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} **{item['name']}**")
    return "\n".join(lines)


def format_single_recipe(recipe: dict) -> str:
    """Format a single recipe compactly."""
    ingredients = ", ".join([f"{ing['quantity']} {ing['name']}" for ing in recipe['ingredients']])
    return f"**{recipe['day']}** | {recipe['theme']} | *{recipe['dish_name']}*\n{ingredients}\n"


def generate_plan_streaming(dietary_pref, theme_requests, additional_prefs):
    """Generate meal plan with streaming output."""
    output = "Generating your meal plan..."
    yield output

    result = generator.generate(
        dietary_preference=dietary_pref,
        theme_requests=theme_requests if theme_requests.strip() else "random",
        additional_preferences=additional_prefs if additional_prefs.strip() else "none"
    )

    if not result["success"]:
        yield f"Error: {result['error']}"
        return

    meal_plan = result["meal_plan"]

    # Show shopping list first
    output = f"## Weekly {meal_plan['dietary_preference'].title()} Meal Plan\n\n"
    output += format_shopping_list(meal_plan) + "\n\n---\n\n"
    output += "## Recipes\n\n"
    yield output
    pytime.sleep(0.1)

    # Stream recipes one by one
    for recipe in meal_plan['recipes']:
        output += format_single_recipe(recipe) + "\n"
        yield output
        pytime.sleep(0.05)

    # Validation status
    validation = run_basic_validation(result)
    status = "All checks passed" if validation["passed"] else f"Issues: {', '.join(validation['errors'])}"
    output += f"---\n*Generated in {result['generation_time']}s | {status}*"
    yield output


def send_email_ui(email, dietary_pref, theme_requests, additional_prefs):
    """Generate and send meal plan via email."""
    if not email or "@" not in email:
        return "Enter a valid email address."

    result = generator.generate(
        dietary_preference=dietary_pref,
        theme_requests=theme_requests if theme_requests.strip() else "random",
        additional_preferences=additional_prefs if additional_prefs.strip() else "none"
    )

    if not result["success"]:
        return f"Generation failed: {result['error']}"

    email_result = send_meal_plan_email(email, result["meal_plan"])
    return email_result["message"]


# Create Gradio interface
with gr.Blocks(title="Weekly Meal Planner") as app:
    gr.Markdown("# Weekly Meal Planner\nGenerate 7 themed dinners with a consolidated shopping list.")

    with gr.Row():
        with gr.Column(scale=1):
            dietary_dropdown = gr.Dropdown(
                choices=["omnivore", "vegetarian", "vegan", "keto"],
                value="omnivore",
                label="Diet"
            )
            theme_input = gr.Textbox(label="Themes (optional)", placeholder="Italian, Mexican...", lines=1)
            additional_input = gr.Textbox(label="Preferences (optional)", placeholder="no nuts, quick meals...", lines=1)
            generate_btn = gr.Button("Generate", variant="primary")

            gr.Markdown("---")
            email_input = gr.Textbox(label="Email", placeholder="you@example.com")
            send_btn = gr.Button("Email Plan")
            email_status = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            output_display = gr.Markdown("Click **Generate** to start.")

    generate_btn.click(
        fn=generate_plan_streaming,
        inputs=[dietary_dropdown, theme_input, additional_input],
        outputs=[output_display]
    )

    send_btn.click(
        fn=send_email_ui,
        inputs=[email_input, dietary_dropdown, theme_input, additional_input],
        outputs=[email_status]
    )

c:\Users\maxke\miniconda3\envs\bootcamp-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Problem #8: Duplicate Generation and Email Plan Mismatch

**Category:** Repeated Code / Functional Bug  
**Priority:** High

**Problem:**  
Both `generate_plan_streaming()` and `send_email_ui()` contain the same meal plan generation logic. When the user clicks "Email Plan", the application generates a new meal plan instead of using the one previously displayed in the Gradio interface.

Since LLM outputs can vary between requests, especially with a temperature of 0.7, the emailed plan may differ from the one the user originally generated. This also creates unnecessary API calls, additional costs, and longer processing times.

**Potential Improvement:**  
Reuse the already generated meal plan when sending an email instead of generating a new one. The shared generation logic could also be simplified to avoid duplicated code and ensure consistent results.

## 9. Launch Application

In [10]:
# Launch the Gradio app
app.launch(share=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
* Running on public URL: https://670bdbc481c7953dfe.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [11]:
app.close()

Closing server running on port: 7860


## 10. Run Test Cases

Runs every test case through the generator and the basic validation checks. This makes 4 real API calls.

In [11]:
TEST_CASES = [
    {"dietary_preference": "omnivore", "theme_requests": "random", "additional_preferences": "none"},
    {"dietary_preference": "vegetarian", "theme_requests": "Italian, Asian, Mexican", "additional_preferences": "no mushrooms"},
    {"dietary_preference": "vegan", "theme_requests": "random", "additional_preferences": "high protein"},
    {"dietary_preference": "keto", "theme_requests": "Mediterranean, American", "additional_preferences": "none"}
]

for case in TEST_CASES:
    result = generator.generate(
        dietary_preference=case["dietary_preference"],
        theme_requests=case.get("theme_requests", "random"),
        additional_preferences=case.get("additional_preferences", "none")
    )
    validation = run_basic_validation(result)
    print(case["dietary_preference"], "->", "PASS" if validation["passed"] else validation["errors"])

omnivore -> PASS
vegetarian -> PASS
vegan -> PASS
keto -> PASS


### Problem #13: Mixed Responsibilities in Test Cases

**Category:** Mixed Concerns / Test Organization  
**Priority:** Low

**Problem:**  
The test cases are defined directly in the same code cell that executes the tests and prints their results. This combines test data, execution logic, and result reporting in one place.

As a result, updating or reusing test scenarios requires modifying the notebook itself. It also makes it harder to maintain a consistent set of test inputs across different testing scripts.

**Potential Improvement:**  
Move the test cases into a separate JSON file and load them when running the tests. This would separate test data from execution logic, improve reusability, and make it easier to update test scenarios without changing the main notebook.

## Refactoring Checklist

### Issues Found:

- [ ] **#1 – Hardcoded Credentials:** Email credentials are stored directly in the configuration cell instead of environment variables.
- [ ] **#2 – Monolithic Function:** `MealPlanGenerator.generate()` handles multiple responsibilities, including API communication, parsing, validation, and error handling.
- [ ] **#3 – Missing Error Handling:** The `recipe_chain.invoke()` API call is outside the `try` block, leaving API-related exceptions unhandled.
- [ ] **#4 – Insufficient Error Information:** Error handling does not consistently identify where failures occur or provide detailed diagnostic information.
- [ ] **#5 – Fragile JSON Processing:** The JSON cleaning logic relies on string splitting and may fail with unexpected LLM response formats.
- [ ] **#6 – Inconsistent Output:** The failure dictionaries do not include the `generation_time` key.
- [ ] **#7 – Duplicated Ingredient Formatting:** `format_meal_plan_email()` and `format_single_recipe()` repeat the same ingredient formatting logic.
- [ ] **#8 – Duplicate Generation:** `send_email_ui()` generates a new meal plan instead of sending the plan already displayed to the user.
- [ ] **#9 – Dead Code:** `SHOPPING_LIST_PROMPT` and `self.shopping_parser` are defined but never used.
- [ ] **#10 – Hardcoded Validation Values:** `run_basic_validation()` uses fixed values of 5 and 7 instead of existing configuration variables.
- [ ] **#11 – Duplicated Validation Logic:** Pydantic models and `run_basic_validation()` check several of the same rules.
- [ ] **#12 – Mixed Email Responsibilities:** `send_meal_plan_email()` combines email construction and SMTP communication.
- [ ] **#13 – Mixed Test Responsibilities:** Test data, test execution, and result reporting are combined in the same notebook cell.

### Priority:

1. **#1 – Hardcoded Credentials:** Prevent potential exposure of sensitive email credentials.
2. **#3 – Missing Error Handling:** Ensure API failures are handled without uncontrolled exceptions.
3. **#8 – Duplicate Generation:** Ensure users receive the same meal plan by email that was previously displayed.

### Prioritization Rationale

I prioritized security first because sensitive credentials should never be exposed through a shared notebook or GitHub repository.

My second priority is improving API error handling to ensure that failures are handled properly and the application remains reliable.

The third priority is fixing duplicate meal plan generation. Users should receive exactly the same meal plan by email that was previously displayed in the application. Generating a new plan could lead to inconsistent results, unnecessary API costs, and a poor user experience.